In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np
import json
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# 1. Setup Model and Tokenizer
# model_name = "BAAI/bge-small-en-v1.5"
model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def get_bge_embeddings(sentences, batch_size=32):
    """
    Generates BGE-M3 embeddings using the transformers library.
    Bypasses multiprocessing bugs by running on the main process.
    """
    all_embeddings = []
    
    # Process in batches for memory efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            
            # BGE-M3 uses the [CLS] token (index 0) for its dense representation
            # Shape: (batch_size, 1024)
            embeddings = outputs.last_hidden_state[:, 0]
            
            # Unit Normalization (L2) - This is CRITICAL for BGE-M3 
            # to spread out the 'Narrow Cone' of Hindi embeddings.
            embeddings = F.normalize(embeddings, p=2, dim=1)
            
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.concatenate(all_embeddings, axis=0)

# --- Usage ---
# embeddings = get_bge_embeddings(clean_for_model, batch_size=32)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:


# ===================== LOAD =====================
with open("relation_results.json") as f:
    relations = json.load(f)

with open("entity_info.json") as f:
    entity_info = json.load(f)

with open("entity_id_dict.json") as f:
    name_to_id = json.load(f)

# ===================== BUILD MAPS =====================
id_to_types = {
    eid: data["types"]
    for eid, data in entity_info.items()
}

entity_to_types = {
    data["name"]: data["types"]
    for data in entity_info.values()
}

# ===================== UNIQUE TRIPLES =====================
unique_triples = set()

for item in relations:
    rel_data = item["relation_data"]

    if rel_data["relation"] == "NONE":
        continue

    unique_triples.add((
        rel_data["source_entity"],
        rel_data["relation"],
        rel_data["target_entity"]
    ))

# ===================== CORE FUNCTION =====================
def get_type_filtered_triples(source_type=None, target_type=None):
    
    rows = []

    for (s, r, t) in unique_triples:

        s_types = entity_to_types.get(s, [])
        t_types = entity_to_types.get(t, [])

        # filter source type
        if source_type is not None and source_type not in s_types:
            continue

        # filter target type
        if target_type is not None and target_type not in t_types:
            continue

        rows.append({
            "source": s,
            "relation": r,
            "target": t,
            "source_types": s_types,
            "target_types": t_types
        })

    return pd.DataFrame(rows)

In [4]:
type1 = "remedy"
type2 = "symptom_physical"
std_rel = "alleviates"

df_fwd = get_type_filtered_triples(type1, type2)
df_rev = get_type_filtered_triples(type2, type1)
print(df_fwd.head())
print(df_rev.head())

    source       relation target  \
0  अभ्यङ्ग  helps_relieve  थकावट   

                                        source_types  \
0  [Medical_Concept, Sanskrit_text, Activity, mun...   

                          target_types  
0  [symptom_physical, Medical_Concept]  
Empty DataFrame
Columns: []
Index: []


In [5]:
for rel in df_fwd['relation']:
    if rel in ['is_a', 'is_an']:
        continue
    sentences = [
        f"this {type1} {' '.join(std_rel.split('_'))} this {type2}",
        f"this {type1} {' '.join(rel.split('_'))} this {type2}"
    ]
    embeddings = get_bge_embeddings(sentences)
    print(rel, ": ", cosine_similarity(embeddings[0].reshape(1, -1), embeddings[1].reshape(1, -1)))
   
# for rel in df_rev['relation']:
#     if rel in ['is_a', 'is_an']:
#         continue
#     sentences = [
#         f"this {type1} {std_rel} this {type2}",
#         f"this {type2} {' '.join(rel.split('_'))} this {type1}"
#     ]
#     embeddings = get_bge_embeddings(sentences)
#     print(rel, ": ", cosine_similarity(embeddings[0].reshape(1, -1), embeddings[1].reshape(1, -1)))
   

helps_relieve :  [[0.9788594]]


(avatar, is_incarnation_of, deity) -> similarity threshold = 0.9

(season, spans, month) -> similarity threshold = 0.85

remedy -> alleviates -> atleast_one(disease, symptom_physical, symptom_mental) 0.9

